## Anomaly Detection

### Loading Data and Packages

In [32]:
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.covariance import EllipticEnvelope

RANDOM_STATE = 42
NAVY = '#1B2F7E'
POINT_BLUE = '#3F7AB5'

def add_border(fig, color=NAVY, width=6):
    fig.patches.append(Rectangle((0, 0), 1, 1, transform=fig.transFigure,
                                 fill=False, edgecolor=color, linewidth=width, zorder=10))

In [33]:
url = "https://raw.githubusercontent.com/Owen-StOnge/IST707PROJECT/main/DataCollectandClean/FinalProject.parquet"
pitchers = pd.read_parquet(url)

pitchers = pitchers.sort_values(["xMLBAMID", "Season"])
g = pitchers.groupby("xMLBAMID")

consecutive = g["Season"].shift(-1) == pitchers["Season"] + 1
pitchers["next_war"] = g["WAR"].shift(-1).where(consecutive)
pitchers["next_money"] = g["Dollars"].shift(-1).where(consecutive)

predictors = ['Age', 'ERA', 'WHIP', 'GB%', 'FIP', 'WAR', 'Zone%', 'K%', 'BB%',
              'Barrel%', 'FBv', 'sp_stuff', 'TBF', "GS"]

new_vars = ["GS", "TBF", "K/9", "BB/9", "K/BB", "H/9", "HR/9", "AVG", "WHIP", "BABIP", "LOB%", "FIP",
            "GB/FB", "LD%", "GB%", "FB%", "IFFB%", "HR/FB", "IFH%", "BUH%", "TTO%", "tERA", "xFIP",
            "WPA", "-WPA", "+WPA", "RE24", "Clutch", "FBv", "O-Swing%", "Z-Swing%", "Swing%",
            "O-Contact%", "Z-Contact%", "Contact%", "Zone%", "F-Strike%", "SwStr%", "CStr%", "C+SwStr%",
            "HRFB%+", "Pull%+", "Cent%+", "Oppo%+", "Soft%+", "Med%+", "Hard%+", "xERA",
            "sp_stuff", "sp_location", "sp_pitching", "EV", "LA", "Barrels", "Barrel%", "maxEV",
            "HardHit", "HardHit%", "EV90"]

#Union of selected predictors + new list (keeps order, drops duplicates)
predictors_full = list(dict.fromkeys(predictors + new_vars))
print('Missing columns:', [c for c in predictors_full if c not in pitchers.columns])
print('Expanded feature count:', len(predictors_full))

#Keep pitchers with at least 5 starts
starters = pitchers[pitchers['GS'] >= 5].copy()

#Standardize every predictor within season (z-score relative to that season's starters)
starters[predictors_full] = (starters.groupby('Season')[predictors_full]
                             .transform(lambda x: (x - x.mean()) / x.std())
                             .replace([np.inf, -np.inf], np.nan))

#Modeling sample: drop rows with no next-season WAR
model_df = starters.dropna(subset=['next_war']).copy()

X = model_df[predictors].copy()
y = model_df['next_war'].copy()
groups = model_df['xMLBAMID'].copy()

print(f'Modeling rows: {len(model_df)}, Unique pitchers: {groups.nunique()}')

Missing columns: []
Expanded feature count: 64
Modeling rows: 2253, Unique pitchers: 722


### Checking for NAs

In [35]:
#Check for missing values 
missing = starters[predictors].isna().sum()
missing_pct = (missing / len(starters) * 100).round(2)

summary = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
print(summary[summary['Missing'] > 0].sort_values('Missing', ascending=False))
print('\nTotal missing values:', int(missing.sum()))
print('Rows with at least one missing value:', int(starters[predictors].isna().any(axis=1).sum()))

          Missing  Percent
sp_stuff     1192    42.19

Total missing values: 1192
Rows with at least one missing value: 1192


In [36]:
#Filling missing values by imputing with the median
non_fbv = [c for c in predictors if c != 'FBv']
imputer = ColumnTransformer([
    ('median', SimpleImputer(strategy='median'), non_fbv),
    ('fbv', SimpleImputer(strategy='constant', fill_value=0), ['FBv']),
])
X_all = pd.DataFrame(imputer.fit_transform(starters[predictors]),
                     columns=non_fbv + ['FBv'], index=starters.index)
print(X_all.shape, 'NaNs left:', int(X_all.isna().sum().sum()))

name_col = next((c for c in ['PlayerName', 'Name', 'player_name'] if c in starters.columns), None)
print('Name column:', name_col)

(2825, 14) NaNs left: 0
Name column: PlayerName


### Anomaly Detectors 

Using 3 different techniques for anomaly detection: isolation forest, PCA reconstruction error, and Fast-MCD

In [39]:
scores = pd.DataFrame(index=starters.index)

#Isolation Forest
iso = IsolationForest(n_estimators=500, random_state=RANDOM_STATE).fit(X_all)
scores['isolation_forest'] = -iso.decision_function(X_all)

#PCA reconstruction error (keep components explaining 90% of variance)
Xs = StandardScaler().fit_transform(X_all)
pca = PCA(n_components=0.90, random_state=RANDOM_STATE).fit(Xs)
recon = pca.inverse_transform(pca.transform(Xs))
scores['pca_recon_error'] = ((Xs - recon) ** 2).sum(axis=1)
print('PCs kept for 90% variance:', pca.n_components_)

#Fast-MCD (robust Mahalanobis distance)
mcd = EllipticEnvelope(contamination=0.05, random_state=RANDOM_STATE).fit(X_all)
scores['mcd_mahalanobis'] = mcd.mahalanobis(X_all)

#Percentile ranks with 1.0 meaning most anomalous and a consensus score
pct = scores.rank(pct=True)
consensus = pct.mean(axis=1)
scored = starters.join(scores).assign(consensus=consensus)
scored['flag_top5'] = scored['consensus'] >= scored['consensus'].quantile(0.95)

print('Flagged as top 5% anomalies:', int(scored['flag_top5'].sum()))

PCs kept for 90% variance: 8
Flagged as top 5% anomalies: 142


In [40]:
#Checking if the methods agree
scores.corr(method='spearman').round(2)

,isolation_forest,pca_recon_error,mcd_mahalanobis
isolation_forest,1.00,0.43,0.78
pca_recon_error,0.43,1.00,0.65
mcd_mahalanobis,0.78,0.65,1.00


In [41]:
#Printing the top 15 most anomalous pitcher-seasons
cols = [name_col, 'Season', 'consensus']
scored.sort_values('consensus', ascending=False).head(15)[cols].round(3)

,PlayerName,Season,consensus
1465,Dillon Overton,2016,0.997
9629,Sean Sullivan,2026,0.996
6259,Connor Seabold,2022,0.994
6494,Jacob deGrom,2023,0.993
3776,Tyler Anderson,2019,0.993
728,Allen Webster,2015,0.992
2984,Chris Tillman,2018,0.990
1854,Eric Skoglund,2017,0.989
6375,Elvin Rodríguez,2022,0.988
733,David Holmberg,2015,0.988


Checking if any of Skubal's previous seasons appear

In [43]:
scored['consensus_rank'] = scored['consensus'].rank(ascending=False).astype(int)

sk = scored[scored[name_col].str.contains('Skubal', case=False, na=False)]
skubal_table = (sk[[name_col, 'Season', 'WAR']]
                .join(pct.loc[sk.index].round(3))
                .join(sk[['consensus', 'consensus_rank']].round(3)))
skubal_table

,PlayerName,Season,WAR,isolation_forest,pca_recon_error,mcd_mahalanobis,consensus,consensus_rank
4439,Tarik Skubal,2020,-1.042916,0.766,0.268,0.679,0.571,1153
4817,Tarik Skubal,2021,-0.470299,0.732,0.851,0.712,0.765,508
5551,Tarik Skubal,2022,0.959746,0.294,0.074,0.061,0.143,2576
6398,Tarik Skubal,2023,1.440002,0.965,0.686,0.771,0.807,379
7241,Tarik Skubal,2024,3.275636,0.963,0.648,0.703,0.771,485
8095,Tarik Skubal,2025,3.770541,0.991,0.872,0.953,0.939,84
8974,Tarik Skubal,2026,2.557753,0.943,0.288,0.655,0.629,926


Skubal's 2025 season was considered an anomaly by all 3 methods. Skubal's 2023, 2024, and 2026 seasons were all scored anomalies by isolation forest and scored moderately or low by PCA reconstruction error and Fast-MCD. This shows his past ability to produce anomalous seasons, which is important to keep in mind when predicting his future and is why high-performing anomalies were left in the data. 